# LSN-002 | From Floating Point to Finite Width
**Maps to: RMD-002 · FR1/FR2 · T-005~T-006**

## Question for this lesson
Values such as `0.1` and `0.9` feel natural in Python, but an FPGA represents numbers with a finite number of bits. How do we map continuous values into finite-width arithmetic?

> Primary new concept: **fixed-point / quantization.**

## Intuition
Think of fixed-point as a ruler with a finite number of marks. `frac_bits` controls how fine the marks are; `total_bits` limits the overall numerical range.

We first simulate this restriction in Python, then require RTL to reproduce the same rules.

In [ ]:
def quantize(x, total_bits=8, frac_bits=4):
    scale = 1 << frac_bits
    min_i = -(1 << (total_bits - 1))
    max_i = (1 << (total_bits - 1)) - 1
    q = round(x * scale)
    q = max(min_i, min(max_i, q))
    return q / scale

for x in [0.1, 0.22, 0.9, 1.7, -0.3]:
    print(x, '->', quantize(x, 8, 4))

In [ ]:
def run_lif_quantized(inputs, total_bits, frac_bits, alpha=0.9, threshold=1.0, reset=0.0):
    q = lambda x: quantize(x, total_bits, frac_bits)
    v = q(0.0)
    spikes = []
    trace = []
    for t, current in enumerate(inputs):
        v = q(q(alpha) * v + q(current))
        spike = v >= q(threshold)
        if spike:
            spikes.append(t)
            v = q(reset)
        trace.append(v)
    return spikes, trace

inputs = [0.22] * 30
for fmt in [(8, 4), (12, 8), (16, 12)]:
    spikes, trace = run_lif_quantized(inputs, *fmt)
    print(f'Q format total={fmt[0]}, frac={fmt[1]} -> spikes {spikes}')

## Observe
- Which parameters change the most as width shrinks?
- Does spike timing change?
- Why can saturation and wraparound create qualitatively different failures?

Do not rush to select a final Q-format. The goal is to see that **numeric representation is itself a design parameter**.

## AI Task
Ask AI to add a parameter sweep, error table, or spike-time comparison. Require it to state the rounding and saturation rules explicitly.

## Human Check
- What problems are solved by integer bits versus fractional bits?
- Why might the accumulator need to be wider than membrane state or weight?
- If two fixed-point implementations are numerically close, must their spike sequences be identical?

## Engineering Handoff
The mature implementation belongs in `python/reference/lif_fixed.py`, while numeric decisions belong in MDD/TDD.

## Exit Ticket
You can explain scale, rounding, and saturation, and demonstrate experimentally how bit width changes neuron behavior.